# State Jobless Claims — Vintage / Point-in-Time Data

**Goal:** Point-in-time weekly **initial claims** and **continued claims** for all US states — no look-ahead bias — for use as features in a predictive model.

**Series (FRED, sourced from DOL ETA):**
- `{ABBR}ICLAIMS` — Initial Claims, weekly, NSA (e.g. `CAICLAIMS`)
- `{ABBR}CCLAIMS` — Continued Claims / Insured Unemployment, weekly, NSA (e.g. `CACCLAIMS`)

**Empirical facts (verified against ALFRED):**
- Full current history goes back to **1986**, but ALFRED vintage archives only begin **Sept 2010**.
- Publication lag from week-ending (Saturday) to FRED capture: **6–19 days, mean ~10**.
- State claims are **essentially never revised** in ALFRED — each value is captured once (~4–6 exceptions per state over 15 years, mostly COVID-era corrections). So within the vintage window, *first release ≈ current value*.
- The critical PIT issue for claims is therefore **availability lag**, not revisions: e.g. on 2020-03-26 the latest available CA observation was the week of **2020-03-07** — the COVID spike (week of 3/21: 186k claims) was not yet visible at state level in FRED.

## 1. Setup

In [ ]:
import os, time, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

FRED_API_KEY = os.getenv("FRED_API_KEY", "12d77a40907e43a92e9a295801db18d2")
FRED_URL     = "https://api.stlouisfed.org/fred/series/observations"

STATES = [
    ("Alabama",              "AL", "01"), ("Alaska",               "AK", "02"),
    ("Arizona",              "AZ", "04"), ("Arkansas",             "AR", "05"),
    ("California",           "CA", "06"), ("Colorado",             "CO", "08"),
    ("Connecticut",          "CT", "09"), ("Delaware",             "DE", "10"),
    ("District of Columbia", "DC", "11"), ("Florida",              "FL", "12"),
    ("Georgia",              "GA", "13"), ("Hawaii",               "HI", "15"),
    ("Idaho",                "ID", "16"), ("Illinois",             "IL", "17"),
    ("Indiana",              "IN", "18"), ("Iowa",                 "IA", "19"),
    ("Kansas",               "KS", "20"), ("Kentucky",             "KY", "21"),
    ("Louisiana",            "LA", "22"), ("Maine",                "ME", "23"),
    ("Maryland",             "MD", "24"), ("Massachusetts",        "MA", "25"),
    ("Michigan",             "MI", "26"), ("Minnesota",            "MN", "27"),
    ("Mississippi",          "MS", "28"), ("Missouri",             "MO", "29"),
    ("Montana",              "MT", "30"), ("Nebraska",             "NE", "31"),
    ("Nevada",               "NV", "32"), ("New Hampshire",        "NH", "33"),
    ("New Jersey",           "NJ", "34"), ("New Mexico",           "NM", "35"),
    ("New York",             "NY", "36"), ("North Carolina",       "NC", "37"),
    ("North Dakota",         "ND", "38"), ("Ohio",                 "OH", "39"),
    ("Oklahoma",             "OK", "40"), ("Oregon",               "OR", "41"),
    ("Pennsylvania",         "PA", "42"), ("Rhode Island",         "RI", "44"),
    ("South Carolina",       "SC", "45"), ("South Dakota",         "SD", "46"),
    ("Tennessee",            "TN", "47"), ("Texas",                "TX", "48"),
    ("Utah",                 "UT", "49"), ("Vermont",              "VT", "50"),
    ("Virginia",             "VA", "51"), ("Washington",           "WA", "53"),
    ("West Virginia",        "WV", "54"), ("Wisconsin",            "WI", "55"),
    ("Wyoming",              "WY", "56"),
]

CLAIM_TYPES = {"iclaims": "ICLAIMS", "cclaims": "CCLAIMS"}

# Note: DOL also publishes Puerto Rico (PR...) and Virgin Islands (VI...) — add to STATES if needed.
print(f"{len(STATES)} states x {len(CLAIM_TYPES)} claim types = {len(STATES)*len(CLAIM_TYPES)} series")

## 2. Fetch vintage data (first release per week)

`output_type=4` returns, per observation week:
- `obs_date` — week ending (Saturday)
- `first_release_date` — when FRED/ALFRED first captured the value (`realtime_start`)
- `superseded_date` — when it was revised away (`9999-12-31` = never, which is almost always the case here)
- `value_first_release`

102 API calls (~2 min with rate-limit-friendly pacing).

In [ ]:
def fred_get(params, retries=4):
    params = {**params, "api_key": FRED_API_KEY, "file_type": "json"}
    for attempt in range(retries):
        data = requests.get(FRED_URL, params=params, timeout=120).json()
        if "error_message" in data:
            if "Too Many Requests" in data["error_message"]:
                time.sleep(2 ** (attempt + 1))
                continue
            raise ValueError(data["error_message"])
        return [o for o in data.get("observations", []) if o["value"] != "."]
    raise RuntimeError(f"Rate limit persists: {params.get('series_id')}")


def fetch_vintage(series_id):
    return fred_get({
        "series_id": series_id,
        "realtime_start": "1776-07-04", "realtime_end": "9999-12-31",
        "output_type": 4, "limit": 100000,
    })


vintage_rows, failed = [], []

for i, (name, abbr, fips) in enumerate(STATES):
    for ctype, suffix in CLAIM_TYPES.items():
        sid = f"{abbr}{suffix}"
        try:
            obs = fetch_vintage(sid)
            for o in obs:
                vintage_rows.append({
                    "state": name, "abbr": abbr, "fips": fips, "claim_type": ctype,
                    "obs_date":            pd.to_datetime(o["date"]),
                    "first_release_date":  pd.to_datetime(o["realtime_start"]),
                    "superseded_date":     o["realtime_end"],
                    "value_first_release": float(o["value"]),
                })
            print(f"[{i+1:2d}/51] {sid:10s} {len(obs)} obs  ({obs[0]['date']} - {obs[-1]['date']})")
        except Exception as e:
            print(f"[{i+1:2d}/51] {sid:10s} FAILED — {e}")
            failed.append(sid)
        time.sleep(0.3)

if failed:
    print(f"\nFailed: {failed}")

df_v = pd.DataFrame(vintage_rows)
df_v["is_current"]   = df_v["superseded_date"] == "9999-12-31"
df_v["pub_lag_days"] = (df_v["first_release_date"] - df_v["obs_date"]).dt.days

print(f"\nVintage rows: {len(df_v):,}  |  States: {df_v['state'].nunique()}")
print(f"Obs range: {df_v['obs_date'].min().date()} - {df_v['obs_date'].max().date()}")
df_v.head()

## 3. Fetch current (full history back to 1986)

The pre-2010 history has no vintages, but since state claims are effectively unrevised, the current values are a reasonable proxy for what was published (with the usual ~10-day lag). Kept separate so you can choose strict-PIT (2010+) or extended history.

In [ ]:
current_rows = []

for i, (name, abbr, fips) in enumerate(STATES):
    for ctype, suffix in CLAIM_TYPES.items():
        sid = f"{abbr}{suffix}"
        try:
            obs = fred_get({"series_id": sid, "observation_start": "1986-01-01", "limit": 100000})
            for o in obs:
                current_rows.append({
                    "state": name, "abbr": abbr, "claim_type": ctype,
                    "obs_date":      pd.to_datetime(o["date"]),
                    "value_current": float(o["value"]),
                })
            print(f"[{i+1:2d}/51] {sid:10s} {len(obs)} obs  latest {obs[-1]['date']} = {float(obs[-1]['value']):,.0f}")
        except Exception as e:
            print(f"[{i+1:2d}/51] {sid:10s} FAILED — {e}")
        time.sleep(0.3)

df_c = pd.DataFrame(current_rows)
print(f"\nCurrent rows: {len(df_c):,}")
df_c.head()

## 4. Revision check

Confirm the "effectively unrevised" claim: join first release vs current and count differences.

In [ ]:
df_rev = df_v.merge(
    df_c[["state", "claim_type", "obs_date", "value_current"]],
    on=["state", "claim_type", "obs_date"], how="inner",
)
df_rev["revision"]     = df_rev["value_current"] - df_rev["value_first_release"]
df_rev["revision_pct"] = df_rev["revision"] / df_rev["value_first_release"] * 100

n_diff = (df_rev["revision"] != 0).sum()
print(f"Rows: {len(df_rev):,}")
print(f"Revised obs: {n_diff} ({n_diff/len(df_rev):.2%}) — claims are effectively unrevised\n")

exceptions = df_rev[df_rev["revision"] != 0].copy()
exceptions["obs_date"] = exceptions["obs_date"].dt.strftime("%Y-%m-%d")
cols = ["state", "claim_type", "obs_date", "value_first_release", "value_current", "revision_pct"]
print("All revision exceptions (largest |%| first):")
print(exceptions.reindex(exceptions["revision_pct"].abs().sort_values(ascending=False).index)[cols]
      .head(30).to_string(index=False))

## 5. Publication lag

In [ ]:
print("Publication lag (days from week-ending Saturday to FRED capture):")
print(df_v.groupby("claim_type")["pub_lag_days"].describe().round(1))

lag_by_state = df_v.groupby(["state","claim_type"])["pub_lag_days"].mean().unstack()
print(f"\nMean lag by state — range across states: "
      f"iclaims {lag_by_state['iclaims'].min():.1f}-{lag_by_state['iclaims'].max():.1f} days, "
      f"cclaims {lag_by_state['cclaims'].min():.1f}-{lag_by_state['cclaims'].max():.1f} days")

## 6. Point-in-time (PIT) feature panel — weekly grid

For each Friday `as_of_date` and each state, take the last `N_LAGS` weekly readings of **both** claim series whose `first_release_date <= as_of_date`.

Implementation note: because releases are append-only, we sort each state's obs by week and use the *running max* of release dates — `searchsorted` then gives the number of weeks available at each as-of date in O(log n). This only counts the contiguous released prefix, which is exactly what "last N consecutive weekly readings" requires.

In [ ]:
N_LAGS = 8  # weekly lags per claim type

AS_OF_GRID = pd.date_range(
    start=(df_v["first_release_date"].min() + pd.offsets.Week(weekday=4)),  # first Friday after archive starts
    end=df_v["first_release_date"].max(),
    freq="W-FRI",
)
print(f"As-of grid: {len(AS_OF_GRID)} Fridays  ({AS_OF_GRID[0].date()} - {AS_OF_GRID[-1].date()})")


def build_pit(vint, claim_type, n_lags, grid):
    """Vectorized PIT extraction for one claim type. Returns dict keyed by (as_of, state)."""
    out = {}
    sub_all = vint[vint["claim_type"] == claim_type]
    for state, g in sub_all.groupby("state", sort=False):
        g = g.sort_values("obs_date")
        rel_eff = g["first_release_date"].cummax().values  # monotone availability frontier
        vals    = g["value_first_release"].values
        obsd    = g["obs_date"].values
        n_avail = np.searchsorted(rel_eff, grid.values, side="right")
        for j, as_of in enumerate(grid):
            n = n_avail[j]
            if n < n_lags:
                continue
            row = {f"{claim_type}_t{k}": vals[n - 1 - k] for k in range(n_lags)}
            row[f"{claim_type}_latest_obs"] = pd.Timestamp(obsd[n - 1])
            row[f"{claim_type}_lag_days"]   = (as_of - pd.Timestamp(obsd[n - 1])).days
            out[(as_of, state)] = row
    return out


pit_i = build_pit(df_v, "iclaims", N_LAGS, AS_OF_GRID)
pit_c = build_pit(df_v, "cclaims", N_LAGS, AS_OF_GRID)

# Combine: keep rows where both claim types are available
keys = sorted(set(pit_i) & set(pit_c))
pit = pd.DataFrame([
    {"as_of_date": k[0], "state": k[1], **pit_i[k], **pit_c[k]} for k in keys
])

# Derived features — the classic claims transforms
for ct in ["iclaims", "cclaims"]:
    lag_cols        = [f"{ct}_t{k}" for k in range(N_LAGS)]
    pit[f"{ct}_4wma"]      = pit[[f"{ct}_t{k}" for k in range(4)]].mean(axis=1)
    pit[f"{ct}_4wma_prev"] = pit[[f"{ct}_t{k}" for k in range(4, 8)]].mean(axis=1)
    pit[f"{ct}_wow_pct"]   = (pit[f"{ct}_t0"] / pit[f"{ct}_t1"] - 1) * 100
    pit[f"{ct}_4wma_pct"]  = (pit[f"{ct}_4wma"] / pit[f"{ct}_4wma_prev"] - 1) * 100

print(f"PIT rows: {len(pit):,}")
print(f"Columns: {list(pit.columns)}")
pit.head(6)

In [ ]:
# Sanity check — the COVID information lag.
# On Friday 2020-03-27 the CA spike week (ending 3/21, 186k claims) was NOT yet in FRED:
chk = pit[(pit["as_of_date"] == "2020-03-27") & (pit["state"] == "California")]
print("As of 2020-03-27 (California):")
print(chk[["iclaims_latest_obs", "iclaims_lag_days", "iclaims_t0", "iclaims_t1"]].to_string(index=False))

chk2 = pit[(pit["as_of_date"] == "2020-04-03") & (pit["state"] == "California")]
print("\nOne week later (2020-04-03) the spike becomes visible:")
print(chk2[["iclaims_latest_obs", "iclaims_lag_days", "iclaims_t0", "iclaims_t1"]].to_string(index=False))

print("\nData staleness at as-of dates (days):")
print(pit[["iclaims_lag_days", "cclaims_lag_days"]].describe().round(1))

## 7. Save outputs

In [ ]:
df_v.drop(columns="superseded_date").to_csv("claims_vintage_first_release.csv", index=False)
df_c.to_csv("claims_current_history.csv", index=False)
pit.to_csv("claims_pit_features.csv", index=False)

print("Saved:")
print("  claims_vintage_first_release.csv — first-release values per week (2010-09+)")
print("  claims_current_history.csv       — full current history back to 1986")
print("  claims_pit_features.csv          — weekly point-in-time ML feature panel")

## 8. Visualizations

In [ ]:
# ── Plot 1: The COVID information lag — what was knowable when ────────────────
ca = df_v[(df_v["state"] == "California") & (df_v["claim_type"] == "iclaims")].sort_values("obs_date")
win = ca[(ca["obs_date"] >= "2020-02-01") & (ca["obs_date"] <= "2020-06-30")]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(win["obs_date"], win["value_first_release"], marker="o", lw=1.5, label="Weekly initial claims (by week ending)")
known = win[win["first_release_date"] <= "2020-03-27"]
ax.plot(known["obs_date"], known["value_first_release"], marker="o", lw=0, ms=10,
        mfc="none", mec="red", mew=2, label="Known in FRED on Fri 2020-03-27")
ax.axvline(pd.Timestamp("2020-03-27"), color="red", ls="--", lw=1, alpha=0.6)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set(title="California initial claims: the point-in-time availability lag (COVID onset)",
       ylabel="Claims (log scale)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Initial claims, 4-week MA, all states (full 1986+ history) ───────
wide_i = (df_c[df_c["claim_type"] == "iclaims"]
          .pivot(index="obs_date", columns="state", values="value_current")
          .sort_index().rolling(4).mean())

fig, ax = plt.subplots(figsize=(13, 6))
for col in wide_i.columns:
    ax.plot(wide_i.index, wide_i[col], lw=0.5, alpha=0.35, color="steelblue")
ax.plot(wide_i.index, wide_i.median(axis=1), lw=2, color="black", label="Median state")
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set(title="Initial Claims by State — 4-week MA, log scale (1986-present)", ylabel="Claims")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Continued claims — selected states ────────────────────────────────
HIGHLIGHT = ["California", "Texas", "New York", "Florida", "Michigan", "Nevada"]
wide_c = (df_c[df_c["claim_type"] == "cclaims"]
          .pivot(index="obs_date", columns="state", values="value_current")
          .sort_index().rolling(4).mean())

fig, ax = plt.subplots(figsize=(13, 5))
for state in HIGHLIGHT:
    ax.plot(wide_c.index, wide_c[state], lw=1.5, label=state)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set(title="Continued Claims (4-week MA, log scale) — selected states", ylabel="Claims")
ax.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 4: Publication lag distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), sharey=True)
for ax, ct in zip(axes, ["iclaims", "cclaims"]):
    lags = df_v.loc[df_v["claim_type"] == ct, "pub_lag_days"]
    ax.hist(lags, bins=range(4, 22), edgecolor="white")
    ax.axvline(lags.mean(), color="red", ls="--", label=f"Mean {lags.mean():.1f}d")
    ax.set(title=f"{ct} — capture lag", xlabel="Days from week-ending to FRED capture")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 5: Latest snapshot — continued claims vs 1 year ago, ranked ──────────
cur = wide_c.dropna(how="all")
yoy = (cur.iloc[-1] / cur.iloc[-53] - 1) * 100  # 4wma now vs ~52 weeks ago
yoy = yoy.sort_values()

fig, ax = plt.subplots(figsize=(8, 14))
colors = plt.cm.RdYlGn_r(plt.Normalize(yoy.min(), yoy.max())(yoy.values))
ax.barh(yoy.index, yoy.values, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set(title=f"Continued claims (4wk MA): % change vs 1 year ago\nas of {cur.index[-1].date()}",
       xlabel="% change YoY")
plt.tight_layout()
plt.show()

## 9. Model-ready feature matrix

Example: predict the **4-week-ahead change in continued claims** from today's PIT claims features.  
Also shown: how to join these weekly features onto the monthly unemployment PIT panel from `bls_state_unemployment_vintages.ipynb`.

In [ ]:
TARGET_HORIZON_WEEKS = 4

# Target: realized (current-data) 4wma of cclaims at as_of + 4 weeks, as % change vs the PIT 4wma
cc = (df_c[df_c["claim_type"] == "cclaims"]
      .pivot(index="obs_date", columns="state", values="value_current")
      .sort_index().rolling(4).mean())
cc_long = cc.stack().rename("cclaims_4wma_realized").reset_index()

tgt = pit[["as_of_date", "state", "cclaims_4wma"]].copy()
tgt["target_week"] = tgt["as_of_date"] + pd.Timedelta(weeks=TARGET_HORIZON_WEEKS)

# align each target_week to the latest obs week <= target_week (merge_asof per state)
tgt = tgt.sort_values("target_week")
cc_long = cc_long.sort_values("obs_date")
tgt = pd.merge_asof(tgt, cc_long, left_on="target_week", right_on="obs_date",
                    by="state", direction="backward")
tgt[f"target_cclaims_chg_{TARGET_HORIZON_WEEKS}w_pct"] = \
    (tgt["cclaims_4wma_realized"] / tgt["cclaims_4wma"] - 1) * 100

model_df = pit.merge(
    tgt[["as_of_date", "state", f"target_cclaims_chg_{TARGET_HORIZON_WEEKS}w_pct"]],
    on=["as_of_date", "state"], how="inner",
).dropna()

drop_cols = ["as_of_date", "state", "iclaims_latest_obs", "cclaims_latest_obs",
             f"target_cclaims_chg_{TARGET_HORIZON_WEEKS}w_pct"]
feature_cols = [c for c in model_df.columns if c not in drop_cols]

print(f"Model-ready rows: {len(model_df):,}")
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Target: target_cclaims_chg_{TARGET_HORIZON_WEEKS}w_pct")

model_df.to_csv("claims_model_features.csv", index=False)
print("\nSaved claims_model_features.csv")
model_df.head(6)

In [ ]:
# Joining with the monthly unemployment PIT panel (from bls_state_unemployment_vintages.ipynb):
# both panels are keyed by (as_of_date, state); take the last weekly claims row per month.
import os
if os.path.exists("pit_features.csv"):
    pit_ur = pd.read_csv("pit_features.csv", parse_dates=["as_of_date"])
    claims_monthly = (pit.assign(as_of_month=pit["as_of_date"].dt.to_period("M"))
                         .sort_values("as_of_date")
                         .groupby(["as_of_month", "state"], as_index=False).last())
    pit_ur["as_of_month"] = pit_ur["as_of_date"].dt.to_period("M")
    combined = pit_ur.merge(
        claims_monthly.drop(columns=["as_of_date"]),
        on=["as_of_month", "state"], how="inner",
    )
    print(f"Combined monthly panel: {len(combined):,} rows, {combined.shape[1]} cols")
    combined.to_csv("combined_pit_features.csv", index=False)
    print("Saved combined_pit_features.csv (unemployment + claims PIT features)")
else:
    print("pit_features.csv not found — run bls_state_unemployment_vintages.ipynb first to build the combined panel.")